# Chunking the Confluence Export -- Step by Step

Put `confluence_export.jsonl` in the same folder as this notebook.

This is the second version of the chunker, adapted for real Confluence-shaped
pages instead of plain markdown. Three real differences from the markdown
version, covered as we go:
1. Body text is HTML now, not markdown -- we cut on `<h4>` tags instead of `##`.
2. Trashed pages are excluded before chunking even starts (mirrors real
   Confluence API behavior, not a shortcut).
3. Category isn't a flat field anymore -- it's derived from the page tree
   (`ancestors`).

In [ ]:
!pip install beautifulsoup4 langchain-text-splitters --quiet

## Step 1: Config (same idea as before)

In [ ]:
MAX_CHUNK_TOKENS = 220        # above this, a section gets force-split
SECONDARY_CHUNK_SIZE = 800    # characters, for the forced split
SECONDARY_OVERLAP = 120       # ~15% overlap, only on forced splits

def token_count(text: str) -> int:
    return round(len(text.split()) * 1.3)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

secondary_splitter = RecursiveCharacterTextSplitter(
    chunk_size=SECONDARY_CHUNK_SIZE,
    chunk_overlap=SECONDARY_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

## Step 2: Load the corpus and look at the raw HTML body

This is the part that's different from the markdown version -- see the
`<h3>`, `<h4>`, `<p>`, `<ol><li>` tags instead of `##`/`###`.

In [ ]:
import json

with open("confluence_export.jsonl") as f:
    pages = [json.loads(line) for line in f]

print(f"Loaded {len(pages)} pages")
print("Status values present:", set(p["status"] for p in pages))

sample = next(p for p in pages if p["title"] == "Network SSL VPN failure")
print("\nRaw HTML body (first 400 chars):")
print(sample["body"]["storage"]["value"][:400])

## Step 3: Exclude trashed pages BEFORE chunking

This isn't us taking a shortcut -- a real Confluence content search
defaults to 'current' status only. A real ingestion pipeline would never
see trashed pages either, so excluding them here matches real behavior.

In [ ]:
retrievable_pages = [p for p in pages if p["status"] != "trashed"]
print(f"{len(pages)} total -> {len(retrievable_pages)} retrievable "
      f"(excluded {len(pages) - len(retrievable_pages)} trashed)")

## Step 4: Split one page's HTML into sections

Cut on `<h4>` tags. The `<h3>` at the very start is skipped as a cut point
-- it's just the page's own title repeated inside the body, not a real
subsection (same reasoning as skipping a document's own title in the
markdown version).

In [ ]:
from bs4 import BeautifulSoup

def split_into_sections(body_html: str, page_title: str):
    soup = BeautifulSoup(body_html, "html.parser")
    sections = []
    current_header = page_title
    current_elements = []

    for el in soup.children:
        tag_name = getattr(el, "name", None)
        if tag_name is None:
            continue                      # skip stray whitespace text nodes
        if tag_name == "h3":
            continue                      # page's own title -- not a real boundary
        if tag_name == "h4":
            if current_elements:
                sections.append((current_header, current_elements))
            current_header = el.get_text().strip()
            current_elements = []
        else:
            current_elements.append(el)

    if current_elements:
        sections.append((current_header, current_elements))
    return sections

# try it on the sample page
sections = split_into_sections(sample["body"]["storage"]["value"], sample["title"])
for header, elements in sections:
    print(f"[{header}] -- {len(elements)} element(s)")

## Step 5: Convert each section's HTML back into clean readable text

`get_text()` alone would run numbered-list items together with no
separator -- so we rebuild the numbering by hand here.

In [ ]:
def section_to_text(elements) -> str:
    lines = []
    for el in elements:
        if el.name == "p":
            text = el.get_text().strip()
            if text:
                lines.append(text)
        elif el.name == "ol":
            for i, li in enumerate(el.find_all("li", recursive=False), start=1):
                lines.append(f"{i}. {li.get_text().strip()}")
        else:
            text = el.get_text().strip()
            if text:
                lines.append(text)
    return "\n".join(lines)

# try it on the first section of the sample page
header, elements = sections[0]
print(f"Section: {header}\n")
print(section_to_text(elements))

## Step 6: `chunk_page()` -- put it all together, with real metadata

Category isn't a flat field on the page -- it's derived from `ancestors`,
the page-tree parent. Labels and read restrictions get carried too, since
those are real fields a production system would want. `doc_type` /
`confuses_with` / `superseded_by` are never referenced here, because they
don't exist on these pages.

In [ ]:
def chunk_page(page: dict) -> list[dict]:
    sections = split_into_sections(page["body"]["storage"]["value"], page["title"])

    category = page["ancestors"][0]["title"] if page["ancestors"] else None
    label_names = [l["name"] for l in page["metadata"]["labels"]["results"]]
    restriction_groups = [
        g["name"] for g in page["restrictions"]["read"]["restrictions"]["group"]["results"]
    ]

    chunks = []
    chunk_idx = 0
    for header_path, elements in sections:
        text = section_to_text(elements)
        if not text:
            continue

        n_tokens = token_count(text)
        if n_tokens <= MAX_CHUNK_TOKENS:
            chunks.append(_make_chunk(page, text, header_path, chunk_idx, n_tokens,
                                       category, label_names, restriction_groups))
            chunk_idx += 1
        else:
            for sub in secondary_splitter.split_text(text):
                chunks.append(_make_chunk(page, sub, header_path, chunk_idx, token_count(sub),
                                           category, label_names, restriction_groups, sub_split=True))
                chunk_idx += 1
    return chunks


def _make_chunk(page, text, header_path, chunk_idx, n_tokens,
                 category, label_names, restriction_groups, sub_split=False):
    return {
        "chunk_id": f"{page['id']}-C{chunk_idx:02d}",
        "page_id": page["id"],
        "title": page["title"],
        "category": category,
        "status": page["status"],
        "version_number": page["version"]["number"],
        "last_updated": page["version"]["when"],
        "labels": label_names,
        "restricted_to": restriction_groups,
        "header_path": header_path,
        "chunk_index": chunk_idx,
        "secondary_split": sub_split,
        "token_count": n_tokens,
        "text": text,
    }

## Step 7: Try it on the sample page and inspect the metadata

In [ ]:
sample_chunks = chunk_page(sample)
for c in sample_chunks:
    print(f"[{c['chunk_id']}] {c['header_path']} (tokens={c['token_count']}, secondary={c['secondary_split']})")
    print(f"   category={c['category']}  status={c['status']}  version={c['version_number']}  "
          f"labels={c['labels']}  restricted_to={c['restricted_to']}")
    print(f"   {c['text'][:100]}...")
    print()

## Step 8: Run it across the whole corpus

In [ ]:
all_chunks = []
for page in retrievable_pages:
    all_chunks.extend(chunk_page(page))

with open("chunks_from_confluence.jsonl", "w") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"{len(retrievable_pages)} pages -> {len(all_chunks)} chunks")
print(f"Avg chunks/page: {len(all_chunks) / len(retrievable_pages):.2f}")
secondary = sum(1 for c in all_chunks if c["secondary_split"])
print(f"Chunks from a forced secondary split: {secondary}")

## Step 9: Confirm no forbidden fields ever made it onto a chunk

This should always print 0 -- doc_type/confuses_with/supersedes/
superseded_by don't exist on the real pages, so there's nothing for the
chunker to leak even by accident.

In [ ]:
forbidden = {"doc_type", "confuses_with", "supersedes", "superseded_by"}
leaked = [c for c in all_chunks if forbidden & c.keys()]
print(f"Chunks with a forbidden field: {len(leaked)} (should be 0)")

## Things to try changing yourself

- Pick a page from the `LABELED` staleness tier (has a `deprecated` label)
  and chunk it -- check that `labels` correctly shows up on every chunk.
- Try removing the `status != "trashed"` filter in Step 3 and see how many
  more chunks you get -- then think about whether a real retriever should
  ever see those.
- Find a Security-category page and check `restricted_to` -- confirm it
  shows `security-team-group` instead of the general `it-department-group`.